In [ ]:
from kafka import KafkaProducer
import time

producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: str(v).encode('utf-8')  # simple string encoding
)

for i in range(10):
    message = f"hello world {i}"
    producer.send('example_topic', value=message)
    print(f"Sent: {message}")
    time.sleep(1)

producer.flush()
producer.close()


## using confluent kafka 


In [ ]:
from confluent_kafka import SerializingProducer
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroSerializer
from confluent_kafka.serialization import StringSerializer, SerializationContext, MessageField
import time

# === Configuration ===

conf = {
    'bootstrap.servers': 'your-cluster.kafka.confluent.cloud:9092',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': 'your_kafka_api_key',
    'sasl.password': 'your_kafka_api_secret',
    'key.serializer': StringSerializer('utf_8'),
    'value.serializer': None  # We'll set Avro serializer below
}

schema_registry_conf = {
    'url': 'https://your-schema-registry-url',
    'basic.auth.user.info': 'your_schema_registry_api_key:your_schema_registry_api_secret'
}

# === Schema Registry Client ===
schema_registry_client = SchemaRegistryClient(schema_registry_conf)

# === Avro Schema ===
avro_schema_str = """
{
  "type": "record",
  "name": "User",
  "fields": [
    {"name": "id", "type": "int"},
    {"name": "name", "type": "string"}
  ]
}
"""

# === Avro Serializer ===
avro_serializer = AvroSerializer(
    schema_registry_client=schema_registry_client,
    schema_str=avro_schema_str,
    to_dict=lambda obj, ctx: obj  # No conversion needed if you already pass dict
)

conf['value.serializer'] = avro_serializer

# === Create Producer ===
producer = SerializingProducer(conf)

topic = "example_topic"

for i in range(10):
    value = {"id": i, "name": f"user_{i}"}
    try:
        producer.produce(
            topic=topic,
            key=str(i),
            value=value,
            on_delivery=lambda err, msg: print(
                f"Delivered: {msg.value()} to {msg.topic()} [{msg.partition()}]" if err is None else f"Delivery error: {err}"
            ),
            context=SerializationContext(topic, MessageField.VALUE)
        )
        producer.poll(0)
    except Exception as e:
        print(f"Failed to send: {e}")
    time.sleep(1)

producer.flush()


In [ ]:
spark=sparkSession.builder.appName('example_session').getOrCreate()
schema=Structtype([
    StructField("name",StringType(),True),
    StructField("place",StringType(),True),
    StructField("Date",TimestampType(),True)
])

df=spark.read.csv('path',inferSchema=True,schema=schema)
df.printSchema()

df1=df.Select(col('name').alias('employee_name'),)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType,DoubleType,StructType,StructField
from pyspark.sql.functions import * 


spark =SparkSession.builder.appName("Abc").getOrCreate()
schema=StructType([
  StructField('product_id',StringType()),
  StructField('product_name',StringType()),
  StructField('original_price',DoubleType()),
  StructField('discount_percentage',DoubleType())
])
df=spark.read.csv('/datasets/products.csv',header=True,inferSchema=True,schema=schema)
df1=df.withColumn('discount_percentage',col('original_price')*col('discount_percentage')*0.01)
display(df)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType,LongType,StructField,StructType
from pyspark.sql.functions import *

spark=SparkSession.builder.appName('abc').getOrCreate()
df=spark.read.json('/datasets/orders.json',multiLine=True)
df1=df.select('customer_id','order_id',explode('products').alias('products'))
df_flat = df.withColumn("product", explode("products")) \
            .select(
                "customer_id",
                "order_id",
                "product.product_name",
                "product.product_price"
            )
display(df_flat)

### Customer Loyalty Score

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructField,StructType,IntegerType

spark =SparkSession.builder.appName("abc").getOrCreate()

rides_data = [
    (1, 101, 1),
    (2, 102, 2),
    (3, 101, 1),
    (4, 103, 3),
    (5, 104, 1),
    (6, 102, 3),
    (7, 101, 2),
    (8, 105, 3),
    (9, 106, 1),
    (10, 107, 2),
    (11, 108, 1),
    (12, 103, 1),
]
rides_columns = ["trip_id", "customer_id", "payment_type_id"]
rides_df = spark.createDataFrame(rides_data, rides_columns)

# Payment types data
payment_types_data = [
    (1, "Card"),
    (2, "Cash"),
    (3, "Ryd credits"),
]
payment_types_columns = ["payment_type_id", "payment_type"]
payment_types_df = spark.createDataFrame(payment_types_data, payment_types_columns)

# Ratings data
ratings_data = [
    (1, 4.5),
    (3, 3.0),
    (4, 5.0),
    (6, 4.0),
    (8, 4.8),
    (10, 3.7),
    (12, 4.1),
]
ratings_columns = ["trip_id", "rating"]
ratings_df = spark.createDataFrame(ratings_data, ratings_columns)

# Customers data
customers_data = [
    (101, "Alice"),
    (102, "Bob"),
    (103, "Carol"),
    (104, "David"),
    (105, "Eva"),
    (106, "Frank"),
    (107, "Grace"),
    (108, "Helen"),
    (109, "Ivy"),
]
customers_columns = ["customer_id", "name"]
customers_df = spark.createDataFrame(customers_data, customers_columns)
abc =rides_df.join(payment_types_df,'payment_type_id','inner').filter(col('payment_type').isin(['Ryd credits','Card']))
xyz=abc.join(ratings_df,'trip_id','inner').select("trip_id", "customer_id", "payment_type", "rating")
xyz=xyz.withColumn("loyalty_score", when(col("rating").isNull(), 10).otherwise(15))
df1=xyz.orderBy(col('loyalty_score').desc()).select('customer_id','loyalty_score')
display(df1)

